In [1]:
#%env JAX_PLATFORM_NAME=cpu
#%env JAX_ENABLE_X64=True
%matplotlib inline
#%matplotlib tk
import numpy as np
import numpy.linalg as lg
import jax
#import jax.numpy as jnp
from util_pred import (load_hur, cov_mat, fit_poly, display_data, fit_poly_dc, validate_pred,
                       ridge_parameter, ridge_opt, avg_replace, avg)
from util import visSphere, visEarth
from morphomatics.manifold import Euclidean, Sphere, PowerManifold
from morphomatics.stats import ExponentialBarycenter, RiemannianRegression
from TimeSeries.verification_metrics import errfun
from TimeSeries.linear_model import RidgeReg as LinRidgeReg
from TimeSeries.linear_model import Reg as LinReg
from TimeSeries.model import Reg, RidgeReg, ARMA, WeightedAverage
from sklearn.covariance import EmpiricalCovariance, ledoit_wolf

deg = 3
Euc, Sph = Euclidean((3,)), Sphere()
dist_euc = jax.jit(Euc.metric.dist)
dist_sph = jax.jit(Sph.metric.dist)
#sph_log = jax.jit(M.metric.log)
err_euc = errfun(Euc.metric.dist)
err_sph = errfun(Sph.metric.dist)
avg = lambda M, x, y: M.metric.geopoint(x, y, .5)
Ex = ExponentialBarycenter()
#mean_b_train = Ex.compute(P, B_train, max_iter=30)

# Load data
H = load_hur(path='../datasets/hur.npz', sph=True, wind=False)
H = [h[:, :3] for h in H]
n_subj, n_trj = len(H), 31
Y = [h[:, :3] for h in H]
#visEarth([Y[2], Y[1]], ['b', 'r'])
#B = np.load('./datasets/deg6_coeff.npz', allow_pickle=True)["arr_0"]
B = np.load('../datasets/deg3_coeff.npz', allow_pickle=True)["arr_0"]
B, Y = B[n_subj-n_trj:n_subj], Y[n_subj-n_trj:n_subj]
n_cp, dim = B[0].shape
# R2 values: Intrinsic
#B, Y_fit = fit_poly_dc(Sph, Y, deg=deg)
##Y = Y_fit

In [2]:
# Sph Prediction
n_learn, n_pred = 1, 2
#n_cp, dim = deg + 1, 3
M = Sphere()
P = PowerManifold(M, n_cp)
dist = jax.jit(M.metric.dist)
err = errfun(M.metric.dist)
n_train = 25
Y_train, B_train, Y_test = Y[:n_train], B[:n_train], Y[n_train:]
n_test = len(Y_test)
# Covariance matrix and mean
Ex = ExponentialBarycenter()
mean_b_train = Ex.compute(P, B_train, max_iter=30)
cov_b_train = cov_mat(P.metric.log, B_train, mean_b_train) #+ 1e-6*np.eye(n_cp*dim)

w = jax.vmap(jax.jit(P.metric.log), (None, 0))(mean_b_train, B_train)
w_vec = w.reshape(n_train, -1)
#cov_b_train, shrinkage = ledoit_wolf(w_vec)

# Compute typical scales
sample_geodesic = P.metric.typicaldist**2  # P.metric.squared_dist(mean_b_train, B_train[0])
sample_mahal = np.linalg.norm(w_vec[0])**2  # Simplified

scale_factor = sample_mahal / sample_geodesic
#print(f"Scale factor: {scale_factor:.2f}")
# Rescale covariance
#cov_b_train = cov_b_train / scale_factor
# Analyze correlation
dists_to_mean = [P.metric.dist(mean_b_train, b) for b in B_train]
eigenvalues = np.sort(np.linalg.eigvalsh(cov_b_train))[::-1]
print(f"\nDistance to mean: {np.mean(dists_to_mean):.4f} ± {np.std(dists_to_mean):.4f}")
print(f"Eigenvalues: {eigenvalues}")
print(f"Variance explained by PC1: {eigenvalues[0]/eigenvalues.sum():.1%}")

def pred(M, Y_test, mu):
    model = RidgeReg(M, mean_b_train, cov_b_train, mu, lag=False, deg=deg)
    Y_p, Y_pred, n_test = [], [], len(Y_test)
    for k in range(n_test):
        y_test = Y_test[k]
        y_p, y_pred = [], []
        for n in range(n_learn, len(y_test)):
            y_learn = y_test[:n]
            #if n >= n_cp: y_learn = y_test[n - n_cp:n]  # faster
            len_x = n + n_pred
            x = np.linspace(0.0, 1.0, len_x)
            model = model.fit(x[:len(y_learn)], y_learn)
            #b = model.trend.control_points
            p = np.array(model.predict(x[n:len_x], iterative=False))
            #p[-1] = M.metric.geopoint(y_learn[-1], p[-1], .25)
            y_pred += [p[1]]
            y_p += [p[0]]
        Y_pred += [np.array(y_pred)]
        Y_p += [np.array(y_p)]
    return Y_p, Y_pred


Distance to mean: 0.7304 ± 0.2797
Eigenvalues: [0.38264895 0.11840928 0.05088007]
Variance explained by PC1: 62.6%


In [6]:
## Predict Sph
Ytest = [Y_test[2]]
mu = 1e-2
Y_p, Y_pred = pred(Sph, Ytest, mu)
mae = [err(Y_p[k], Ytest[k][1:]).mae() for k in range(len(Ytest))]
print('MAE 6h: {:.4f}'.format(np.mean(mae)))
Y_pred = [y[:-1] for y in Y_pred]
mae = [err(Y_pred[k], Ytest[k][2:]).mae() for k in range(len(Ytest))]
print('MAE 12h: {:.4f}'.format(np.mean(mae)))

MAE 6h: 0.0280
MAE 12h: 0.0380


In [7]:
visEarth([Ytest[0], Y_pred[0]], ['b', 'r'])


C:\Users\essi\PycharmProjects\pred\util.py:204: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  sc = m.scatter(x, y, marker=marker, linewidth=linewidth, s=s, c=color, cmap=c_map, norm=norm)


In [3]:
# R2 values: Intrinsic
B, Y_fit = fit_poly_dc(Sph, Y, deg=deg)
Y_mean = [Ex.compute(Sph, y) for y in Y]
x = [err_euc(Y_fit[k],Y[k]).r2(Y_mean[k]) for k in range(n_trj)]
rr=RiemannianRegression(Sph,Y[1],np.linspace(0.0, 1.0, len(Y[1])),deg)
R2_sph = []
for j in range(n_trj):
    y, yfit, ymean = Y[j], Y_fit[j], Y_mean[j]
    a = np.sum([dist_sph(y[k], yfit[k]) ** 2 for k in range(len(y))])
    b = np.sum([dist_sph(ymean, y[k]) ** 2 for k in range(len(y))])
    R2_sph.append(1 - a / b)

print(np.mean(R2_sph))

0.9769413158849061


In [ ]:
# R2 values: Extrinsic
B, Y_fit = fit_poly_dc(Euc, Y, deg=deg)
Y_fit = [y/lg.norm(y) for y in Y]
#Y_mean = [np.mean(y, axis=0)/lg.norm(np.mean(y, axis=0)) for y in Y]
Y_mean = [Ex.compute(Sph, y) for y in Y]
x = [err_euc(Y_fit[k],Y[k]).r2(Y_mean[k]) for k in range(n_trj)]
# R2_euc = []
# for j in range(n_trj):
#     y, yfit, ymean = Y[j], Y_fit[j], Y_mean[j]
#     #a = np.sum([dist_euc(y[k], yfit[k]) ** 2 for k in range(len(y))])
#     #b = np.sum([dist_euc(ymean, y[k]) ** 2 for k in range(len(y))])
#     a = np.sum([dist_sph(y[k], yfit[k]) ** 2 for k in range(len(y))])
#     b = np.sum([dist_sph(ymean, y[k]) ** 2 for k in range(len(y))])
#     R2_euc.append(1 - a / b)
#
# print(np.mean(R2_euc))